In [ ]:
import random
import json
from pathlib import Path

In [ ]:
EPSILON = "ε"

In [ ]:
class NFA:
    def __init__(self, states, alphabet, transitions, start_state, accept_states):
        self.states = states
        self.alphabet = alphabet
        self.transitions = transitions or {}
        self.start_state = start_state
        self.accept_states = accept_states

    def add_transition(self, from_state, input_symbol, to_state):
        if from_state not in self.transitions:
            self.transitions[from_state] = {}
        if input_symbol not in self.transitions[from_state]:
            self.transitions[from_state][input_symbol] = set()
        self.transitions[from_state][input_symbol].add(to_state)

    def get_transitions(self, state, input_symbol):
        if state in self.transitions and input_symbol in self.transitions[state]:
            return self.transitions[state][input_symbol]
        return set()

    def is_accepting(self, state):
        return state in self.accept_states

In [ ]:
def initialize_nfa(num_states, rng):
    if num_states < 3:
        raise ValueError("Number of states must be at least 3.")

    states = set(range(num_states))
    alphabet = {"0", "1"}

    start_state = 0

    num_accept_states = rng.randint(1, 3)

    accept_states = set(
        rng.sample(list(states), num_accept_states)
    )

    nfa = NFA(
        states=states,
        alphabet=alphabet,
        transitions={},
        start_state=start_state,
        accept_states=accept_states
    )

    return nfa

In [ ]:
def get_reachable_states(nfa, start_state = 0):
    reachable = set()
    stack = [start_state]

    while stack:
        current_state = stack.pop()
        if current_state not in reachable:
            reachable.add(current_state)
            for symbol in nfa.alphabet | {EPSILON}:
                for next_state in nfa.get_transitions(current_state, symbol):
                    stack.append(next_state)

    return reachable

In [ ]:
def generate_random_transitions(nfa, min_transitions_per_state, max_transitions_per_state, rng, allow_epsilon):
    for state in nfa.states:
        num_transitions = rng.randint(min_transitions_per_state, max_transitions_per_state)
        for _ in range(num_transitions):
            input_symbol = rng.choice(list(nfa.alphabet))
            to_state = rng.choice(list(nfa.states))
            nfa.add_transition(state, input_symbol, to_state)

    if allow_epsilon:
        num_epsilon_transitions = rng.randint(2, 4)
        for _ in range(num_epsilon_transitions):
            from_state = rng.choice(list(nfa.states))
            to_state = rng.choice(list(nfa.states - {from_state}))
            nfa.add_transition(from_state, EPSILON, to_state)

In [ ]:
def all_states_reachable(nfa):
    reachable_states = get_reachable_states(nfa, nfa.start_state)

    return reachable_states == nfa.states

In [ ]:
def generate_nfa(seed= 0, num_states=10, min_transitions_per_state=1, max_transitions_per_state=3, allow_epsilon=False):
    rng = random.Random(seed)
    max_attempts = 1000

    for _ in range(max_attempts):
        nfa = initialize_nfa(num_states, rng)
        generate_random_transitions(
            nfa, 
            min_transitions_per_state,
            max_transitions_per_state, 
            rng, 
            allow_epsilon
        )

        if all_states_reachable(nfa):
            return nfa

    return RuntimeError("Could not generate a fully reachable NFA.")

In [ ]:
def get_nfa_signature(nfa):
    transitions = frozenset(
        (from_state, symbol, to_state)
        for from_state, symbol_dict in nfa.transitions.items()
        for symbol, destinations in symbol_dict.items()
        for to_state in destinations
    )

    return (
        frozenset(nfa.states),
        frozenset(nfa.alphabet),
        nfa.start_state,
        frozenset(nfa.accept_states),
        transitions
    )

## Generate Sparse NFA (upto 3 transitions per state)

In [ ]:
sparse_nfas = []
sparse_nfa_signatures = set()

counter = 0
seed = 0

while counter < 50:
    nfa = generate_nfa(seed=seed, num_states=10, min_transitions_per_state=1, max_transitions_per_state=3, allow_epsilon=False)
    signature = get_nfa_signature(nfa)

    if signature not in sparse_nfa_signatures:
        sparse_nfas.append(nfa)
        sparse_nfa_signatures.add(signature)
        counter += 1

    seed += 1


In [ ]:
sparse_nfas[0].transitions

{0: {'1': {3, 5, 9}},
 1: {'0': {0}},
 2: {'0': {2}, '1': {1, 8}},
 3: {'1': {2, 9}, '0': {7}},
 4: {'1': {4, 5}, '0': {8}},
 5: {'0': {5, 7, 8}},
 6: {'0': {3}, '1': {5}},
 7: {'1': {5, 6}, '0': {4}},
 8: {'0': {0, 3, 5}},
 9: {'1': {6}}}

## Generate Dense NFA (upto 7 transitions per state)

In [ ]:
dense_nfas = []
dense_nfa_signatures = set()

counter = 0
seed = 0

while counter < 50:
    nfa = generate_nfa(seed=seed, num_states=10, min_transitions_per_state=4, max_transitions_per_state=7, allow_epsilon=False)
    signature = get_nfa_signature(nfa)

    if signature not in dense_nfa_signatures:
        dense_nfas.append(nfa)
        dense_nfa_signatures.add(signature)
        counter += 1

    seed += 1


In [ ]:
dense_nfas[0].transitions

{0: {'1': {6, 7, 9}, '0': {1, 4, 8}},
 1: {'0': {1, 4, 5, 8}, '1': {5, 7}},
 2: {'1': {0, 5, 8}, '0': {1, 3, 5}},
 3: {'0': {2, 5}, '1': {1, 8}},
 4: {'0': {8, 9}, '1': {5, 7, 8}},
 5: {'1': {2, 7}, '0': {1, 2, 9}},
 6: {'0': {0, 3, 8}, '1': {8}},
 7: {'1': {1, 3, 5, 7}, '0': {0, 5, 7}},
 8: {'1': {2, 6}, '0': {1, 3, 9}},
 9: {'0': {1, 5, 6, 9}}}

## Generate NFA with Epsilon Transition

In [ ]:
epsilon_nfas = []
epsilon_nfa_signatures = set()

counter = 0
seed = 0

while counter < 50:
    nfa = generate_nfa(seed=seed, num_states=10, min_transitions_per_state=1, max_transitions_per_state=5, allow_epsilon=True)
    signature = get_nfa_signature(nfa)

    if signature not in epsilon_nfa_signatures:
        epsilon_nfas.append(nfa)
        epsilon_nfa_signatures.add(signature)
        counter += 1

    seed += 1

In [ ]:
epsilon_nfas[10].transitions

{0: {'0': {3, 8}, '1': {2, 5, 7}},
 1: {'0': {5}},
 2: {'1': {2}},
 3: {'1': {2, 4, 6, 7}, '0': {4}},
 4: {'0': {3, 7}, '1': {0}, 'ε': {2}},
 5: {'1': {3, 6, 8}},
 6: {'0': {3, 9}, '1': {3, 8}, 'ε': {7}},
 7: {'0': {7}, 'ε': {3}},
 8: {'0': {2, 8}, '1': {9}},
 9: {'0': {1, 3, 7}}}

In [ ]:
def nfa_to_dict(nfa):
    return {
        "states": sorted(nfa.states),
        "alphabet": sorted(nfa.alphabet),
        "start_state": nfa.start_state,
        "accept_states": sorted(nfa.accept_states),
        "transitions": {
            str(state): {
                symbol: sorted(destinations)
                for symbol, destinations in symbol_dict.items()
            }
            for state, symbol_dict in nfa.transitions.items()
        }
    }

## Exponential-BlowUp NFAs

In [ ]:
def generate_blowup_nfas(min_states, max_states):
    rng = random.Random(0)
    blow_up_nfas = []

    for n in range(min_states, max_states + 1):
        blow_up_nfa = initialize_nfa(n, rng)

        blow_up_nfa.add_transition(0, "0", 0)
        blow_up_nfa.add_transition(0, "1", 0)
        blow_up_nfa.add_transition(0, "1", 1)
        for i in range(1, n):
            blow_up_nfa.add_transition(i, "0", i + 1)
            blow_up_nfa.add_transition(i, "1", i + 1)

        blow_up_nfa.accept_states = {n}

        blow_up_nfas.append(blow_up_nfa)

    return blow_up_nfas

In [ ]:
blowup_nfas = generate_blowup_nfas(4, 10)

## Save NFAs

In [ ]:
output_dir = Path("data/generated")
output_dir.mkdir(parents=True, exist_ok=True)

data = [nfa_to_dict(nfa) for nfa in sparse_nfas]

with open(output_dir / "sparse_nfas.json", "w") as file:
    json.dump(data, file, indent=4)

data = [nfa_to_dict(nfa) for nfa in dense_nfas]

with open(output_dir / "dense_nfas.json", "w") as file:
    json.dump(data, file, indent=4)

data = [nfa_to_dict(nfa) for nfa in epsilon_nfas]

with open(output_dir / "epsilon_nfas.json", "w") as file:
    json.dump(data, file, indent=4)

with open(output_dir / "blowup_nfas.json", "w") as file:
    json.dump([nfa_to_dict(nfa) for nfa in blowup_nfas], file, indent=4)

## NFA to DFA

In [ ]:
test_nfa = NFA(
    states={0, 1, 2, 3},
    alphabet={"0", "1"},
    transitions={},
    start_state=0,
    accept_states={3}
)

test_nfa.add_transition(0, EPSILON, 1)
test_nfa.add_transition(1, EPSILON, 2)
test_nfa.add_transition(2, "0", 3)
test_nfa.add_transition(3, "1", 3)

In [ ]:
def epsilon_closure(nfa, states):
    closure = set(states)
    stack = list(states)

    while stack:
        current_state = stack.pop()
        for next_state in nfa.get_transitions(current_state, EPSILON):
            if next_state not in closure:
                closure.add(next_state)
                stack.append(next_state)

    return closure

In [ ]:
def move(nfa, states, input_symbol):
    next_states = set()
    for state in states:
        next_states.update(nfa.get_transitions(state, input_symbol))
    return next_states

In [ ]:
class DFA:
    def __init__(self, states, alphabet, start_state, accept_states):
        self.states = states
        self.alphabet = alphabet
        self.start_state = start_state
        self.accept_states = accept_states
        self.transitions = {}

    def add_transition(self, from_state, input_symbol, to_state):
        if from_state not in self.states or to_state not in self.states:
            raise ValueError("Invalid state.")

        if input_symbol not in self.alphabet:
            raise ValueError("Invalid input symbol.")

        if from_state not in self.transitions:
            self.transitions[from_state] = {}

        self.transitions[from_state][input_symbol] = to_state

    def get_transition(self, state, input_symbol):
        return self.transitions.get(state, {}).get(input_symbol)

In [ ]:
from collections import deque

def nfa_to_dfa(nfa):
    start_state = frozenset(
        epsilon_closure(nfa, {nfa.start_state})
    )

    dfa_states = {start_state}
    dfa_accept_states = set()
    dfa_transitions = {}

    queue = deque([start_state])

    while queue:
        current_state = queue.popleft()

        if current_state & nfa.accept_states:
            dfa_accept_states.add(current_state)

        for symbol in nfa.alphabet:
            next_state = epsilon_closure(
                nfa,
                move(
                    nfa,
                    epsilon_closure(nfa, current_state),
                    symbol
                )
            )

            next_state = frozenset(next_state)

            if next_state not in dfa_states:
                dfa_states.add(next_state)
                queue.append(next_state)

            if current_state not in dfa_transitions:
                dfa_transitions[current_state] = {}

            dfa_transitions[current_state][symbol] = next_state

    dfa = DFA(
        states=dfa_states,
        alphabet=nfa.alphabet,
        start_state=start_state,
        accept_states=dfa_accept_states
    )

    dfa.transitions = dfa_transitions

    return dfa

In [ ]:
test_dfa = nfa_to_dfa(test_nfa)

In [ ]:
test_dfa.start_state

frozenset({0, 1, 2})

In [ ]:
test_dfa.accept_states

{frozenset({3})}

In [ ]:
test_dfa.alphabet

{'0', '1'}

In [ ]:
test_dfa.states

{frozenset(), frozenset({3}), frozenset({0, 1, 2})}

In [ ]:
test_dfa.transitions

{frozenset({0, 1, 2}): {'0': frozenset({3}), '1': frozenset()},
 frozenset({3}): {'0': frozenset(), '1': frozenset({3})},
 frozenset(): {'0': frozenset(), '1': frozenset()}}

In [ ]:
sparse_dfas = [nfa_to_dfa(nfa) for nfa in sparse_nfas]

dense_dfas = [nfa_to_dfa(nfa) for nfa in dense_nfas]

epsilon_dfas = [nfa_to_dfa(nfa) for nfa in epsilon_nfas]

blowup_dfas = [nfa_to_dfa(nfa) for nfa in blowup_nfas]

In [ ]:
sparse_dfa_states = [len(dfa.states) for dfa in sparse_dfas]
dense_dfa_states = [len(dfa.states) for dfa in dense_dfas]
epsilon_dfa_states = [len(dfa.states) for dfa in epsilon_dfas]
blowup_dfa_states = [len(dfa.states) for dfa in blowup_dfas]

print("Sparse:", sparse_dfa_states)
print("Dense:", dense_dfa_states)
print("Epsilon:", epsilon_dfa_states)
print("Blowup:", blowup_dfa_states)


print("Average Sparse DFA states:", sum(sparse_dfa_states) / len(sparse_dfa_states))
print("Average Dense DFA states:", sum(dense_dfa_states) / len(dense_dfa_states))
print("Average Epsilon DFA states:", sum(epsilon_dfa_states) / len(epsilon_dfa_states))
print("Average Blowup DFA states:", sum(blowup_dfa_states) / len(blowup_dfa_states))

Sparse: [37, 37, 31, 40, 24, 39, 19, 32, 44, 19, 60, 54, 34, 14, 60, 30, 37, 24, 58, 18, 37, 51, 38, 22, 15, 31, 30, 20, 70, 65, 62, 19, 16, 34, 43, 43, 63, 34, 51, 45, 26, 26, 15, 39, 35, 40, 17, 29, 22, 42]
Dense: [19, 10, 26, 25, 24, 24, 26, 11, 31, 32, 14, 15, 17, 9, 13, 11, 15, 20, 31, 17, 24, 18, 38, 19, 17, 25, 17, 22, 17, 28, 18, 42, 39, 18, 28, 13, 18, 18, 21, 22, 21, 23, 17, 24, 9, 15, 10, 11, 17, 13]
Epsilon: [15, 26, 18, 14, 21, 19, 21, 14, 16, 19, 25, 10, 27, 27, 20, 16, 32, 43, 22, 17, 27, 21, 13, 29, 26, 42, 23, 39, 12, 17, 13, 22, 18, 21, 14, 7, 10, 23, 12, 50, 23, 29, 24, 27, 30, 11, 20, 9, 15, 18]
Blowup: [16, 32, 64, 128, 256, 512, 1024]
Average Sparse DFA states: 35.82
Average Dense DFA states: 20.24
Average Epsilon DFA states: 21.34
Average Blowup DFA states: 290.2857142857143


## Minimize DFA

In [ ]:
# DFA minimization via Moore's partition refinement:
# E. F. Moore, "Gedanken-experiments on sequential machines", Automata Studies, 1956.

def initial_partition(dfa):
    # Accepting and non-accepting states are distinguishable by the empty string
    return {state: int(state in dfa.accept_states) for state in dfa.states}

In [ ]:
def refine_partition(dfa, block_of):
    # Split blocks: states stay together only if every symbol leads to the same block
    symbols = sorted(dfa.alphabet)
    block_ids = {}
    new_block_of = {}

    for state in dfa.states:
        key = (
            block_of[state],
            tuple(block_of[dfa.get_transition(state, symbol)] for symbol in symbols)
        )

        if key not in block_ids:
            block_ids[key] = len(block_ids)

        new_block_of[state] = block_ids[key]

    return new_block_of

In [ ]:
def minimize_dfa(dfa):
    block_of = initial_partition(dfa)
    num_blocks = len(set(block_of.values()))

    # Refinement only splits blocks, so an unchanged block count means a fixed point
    while True:
        block_of = refine_partition(dfa, block_of)
        new_num_blocks = len(set(block_of.values()))

        if new_num_blocks == num_blocks:
            break

        num_blocks = new_num_blocks

    # Each block of equivalent states becomes one state of the minimized DFA
    min_dfa = DFA(
        states=set(block_of.values()),
        alphabet=dfa.alphabet,
        start_state=block_of[dfa.start_state],
        accept_states={block_of[state] for state in dfa.accept_states}
    )

    # Equivalent states agree on successor blocks, so repeated writes are identical
    for state in dfa.states:
        for symbol in dfa.alphabet:
            min_dfa.add_transition(
                block_of[state],
                symbol,
                block_of[dfa.get_transition(state, symbol)]
            )

    return min_dfa

In [ ]:
# Sanity-check helpers: compare two DFAs on random strings (a spot check, not a proof)

def dfa_accepts(dfa, string):
    state = dfa.start_state

    for symbol in string:
        state = dfa.get_transition(state, symbol)

    return state in dfa.accept_states


def same_language(dfa_a, dfa_b, rng, num_strings=500, max_length=20):
    for _ in range(num_strings):
        length = rng.randint(0, max_length)
        string = "".join(rng.choice(sorted(dfa_a.alphabet)) for _ in range(length))

        if dfa_accepts(dfa_a, string) != dfa_accepts(dfa_b, string):
            return False

    return True

In [ ]:
sparse_min_dfas = [minimize_dfa(dfa) for dfa in sparse_dfas]

dense_min_dfas = [minimize_dfa(dfa) for dfa in dense_dfas]

epsilon_min_dfas = [minimize_dfa(dfa) for dfa in epsilon_dfas]

blowup_min_dfas = [minimize_dfa(dfa) for dfa in blowup_dfas]

In [ ]:
rng = random.Random(0)

for name, dfas, min_dfas in [
    ("Sparse", sparse_dfas, sparse_min_dfas),
    ("Dense", dense_dfas, dense_min_dfas),
    ("Epsilon", epsilon_dfas, epsilon_min_dfas),
    ("Blowup", blowup_dfas, blowup_min_dfas),
]:
    assert all(same_language(dfa, min_dfa, rng) for dfa, min_dfa in zip(dfas, min_dfas))
    print(name, "minimized DFAs match originals on random strings")

sparse_min_dfa_states = [len(dfa.states) for dfa in sparse_min_dfas]
dense_min_dfa_states = [len(dfa.states) for dfa in dense_min_dfas]
epsilon_min_dfa_states = [len(dfa.states) for dfa in epsilon_min_dfas]
blowup_min_dfa_states = [len(dfa.states) for dfa in blowup_min_dfas]

print()
print("Sparse:", sparse_min_dfa_states)
print("Dense:", dense_min_dfa_states)
print("Epsilon:", epsilon_min_dfa_states)
print("Blowup:", blowup_min_dfa_states)
print()

for name, before, after in [
    ("Sparse", sparse_dfa_states, sparse_min_dfa_states),
    ("Dense", dense_dfa_states, dense_min_dfa_states),
    ("Epsilon", epsilon_dfa_states, epsilon_min_dfa_states),
    ("Blowup", blowup_dfa_states, blowup_min_dfa_states),
]:
    avg_before = sum(before) / len(before)
    avg_after = sum(after) / len(after)
    reduction = 100 * (1 - avg_after / avg_before)
    print(f"{name}: avg {avg_before:.2f} -> {avg_after:.2f} states ({reduction:.1f}% reduction)")